## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the Dataset

In [ ]:
df = pd.read_csv("../00-dataSource/raw_data.csv")

## 3. Initial Data Exploration

In [ ]:
# Show first 5 rows: quick preview of dataset structure and values
print("First 5 rows:")
display(df.head())

# Show dataset size: number of rows and columns
print("Shape of dataset:")
print(df.shape)

# Show column names: list of all features
print("\nColumn names:")
print(df.columns)

# Show index: how rows are labeled (usually default range)
print("\nIndex:")
print(df.index)

# Show numerical summary: mean, min, max, etc.
print("\nStatistical summary (Numerical):")
display(df.describe())

# Show data types and non-null counts: detect missing values & wrong types
print("\nDataset information:")
df.info()

# Show missing values per column: identify incomplete data
print("\nMissing values:")
display(df.isnull().sum())

# Show number of duplicate rows: detect repeated records
print("\nDuplicate rows:")
print(df.duplicated().sum())

### Summary of the Dataset

From the initial exploration, the dataset contains **3500 rows** and **24 columns**, meaning there are 3500 observations and 24 features describing digital lifestyle, personal characteristics, and mental well-being.


### Dataset Structure

| Category              | Details |
|----------------------|--------|
| Number of Rows       | 3500 |
| Number of Columns    | 24 |
| Numerical Features   | 18 |
| Categorical Features | 6 |
| Missing Values       | None |
| Duplicate Rows       | 0 |

The dataset includes both **numerical** and **categorical** variables. Numerical features represent measurable values such as screen time, sleep hours, and scores, while categorical features represent grouped information such as gender, region, and device type.


### Feature Types

- **Numerical Features**:
  - age
  - device_hours_per_day
  - phone_unlocks
  - notifications_per_day
  - social_media_mins
  - study_mins
  - sleep_hours
  - stress_level
  - productivity_score
  - and other score-based variables

- **Categorical Features**:
  - gender
  - region
  - income_level
  - education_level
  - daily_role
  - device_type


### Key Observations

- The dataset is **complete**, with no missing values in any column.
- There are **no duplicate rows**, meaning each record is unique.
- The data types are correctly separated into:
  - numerical (`int64`, `float64`)
  - categorical (`str`)
- The dataset covers a wide range of features related to:
  - digital usage behavior
  - mental health indicators
  - productivity and lifestyle factors



### Statistical Insights

| Feature                    | Mean | Min | Max |
|---------------------------|------|-----|-----|
| Age                       | 28.08 | 13 | 50 |
| Device Hours per Day      | 7.32 | 0.28 | 17.16 |
| Phone Unlocks             | 147 | 9 | 374 |
| Notifications per Day     | 335 | 22 | 1211 |
| Sleep Hours               | 7.25 | 3 | 11 |
| Stress Level              | 5.08 | 1 | 10 |
| Productivity Score        | 65.30 | 33 | 95 |
| Digital Dependence Score  | 36.68 | 5.6 | 89.2 |

These statistics indicate:
- Users spend an average of **7.3 hours on devices daily**
- Sleep averages around **7.25 hours**
- Stress levels range from **low to high (1–10 scale)**
- Productivity scores are moderately distributed



## 4. Data Cleaning

### 4.1 Handling Missing Values

All columns in the dataset contain **0 missing values**, which means the dataset is complete.

No imputation or removal is required for missing data.

In [ ]:
# Check missing values in each column
print("Missing values:")
display(df.isnull().sum())

### 4.2 Handling Duplicate Records

There are **no duplicate rows** in the dataset.

This ensures that each record is unique and no bias is introduced from repeated data.

In [ ]:
# Check duplicate rows
print("Duplicate rows:")
print(df.duplicated().sum())

### 4.3 Data Type Verification

The dataset contains:
- numerical data (int64, float64)
- categorical data (string)

The data types appear correct and suitable for analysis. No conversion is required at this stage.

In [ ]:
# Check data types
print("Data types:")
df.dtypes

### 4.4 Check Logical Consistency

we divide it into two
- Categorical feature(Text, String)
- Numerical feature 

In [ ]:
# categorical feature

# check unique value 
for col in ['gender','region','income_level','education_level','daily_role','device_type']:
    print(col, df[col].unique())

# check inconsistent format 
for col in ['gender','region','income_level','education_level','daily_role','device_type']:
    df[col] = df[col].str.strip().str.lower()

In [ ]:
# numerical feature

# checking hours: hours cannot exceed 24 hours and could not be negative value 
(df['device_hours_per_day'] < 0).sum()
(df['device_hours_per_day'] > 24).sum()
(df['notifications_per_day'] < 0).sum()
(df['social_media_mins'] < 0).sum()
(df['study_mins'] < 0).sum()
(df['sleep_hours'] < 0).sum()
(df['sleep_hours'] > 24).sum()

# checking day should be in between 0 to 7
(df['physical_activity_days'] < 0).sum()
(df['physical_activity_days'] > 7).sum()

# score should be in range 0 - 10
(df['stress_level'] < 0).sum()
(df['stress_level'] > 10).sum()

# productivity should be in range 0 - 100
(df['productivity_score'] < 0).sum()
(df['productivity_score'] > 100).sum()


### 4.5 remove some features that are not important

Several features were removed to improve model performance and avoid data leakage.

- The id column was removed as it does not provide useful information.
- Anxiety, depression, and happiness scores were removed because they are strongly related to stress level.
- Focus score was removed as it directly relates to productivity.
- High risk flag was removed as it may be derived from stress-related features.

The remaining features include relevant behavioral and contextual variables for prediction.

In [ ]:
df = df.drop(columns=[
    'id',
    'anxiety_score',
    'depression_score',
    'happiness_score',
    'focus_score',
    'high_risk_flag'
])

### 4.6 Outlier Detection

In this step, we detect outliers in numerical features.

Outliers are extreme values that differ significantly from other observations.

They may represent:
- real but rare behavior
- data entry errors

Outliers are detected using the IQR (Interquartile Range) method.

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Function to count outliers
def count_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return ((df[col] < lower) | (df[col] > upper)).sum()

# Check outliers for each column
for col in num_cols:
    print(f"{col}: {count_outliers(df, col)} outliers")

In [ ]:
# Remove impossible values only

df = df[df['device_hours_per_day'] <= 24]
df = df[df['sleep_hours'] <= 24]
df = df[df['physical_activity_days'] <= 7]

### Outlier Detection Summary

Outliers were identified using the IQR method across numerical features.

| Feature                   | Outliers |
|---------------------------|----------|
| device_hours_per_day      | 32 |
| phone_unlocks             | 16 |
| notifications_per_day     | 189 |
| social_media_mins         | 238 |
| study_mins                | 11 |
| sleep_hours               | 20 |
| productivity_score        | 52 |
| digital_dependence_score  | 33 |

Some features such as age, physical activity days, sleep quality, and stress level showed no outliers.


### Interpretation & Decision

Outliers are mainly found in behavioral features such as device usage and social media activity.

These values are **not removed** because:
- they represent **realistic human behavior**
- they provide **important signals for the model**
- no invalid or impossible values were detected

Removing them may lead to loss of meaningful information and reduce model performance.


## Copy clean data to csv

In [ ]:
# Save cleaned dataset
df.to_csv("../00-dataSource/cleaned_data.csv", index=False)

print("Cleaned dataset saved successfully!")